In [4]:
import requests
import time
import random
import bs4
import base64
import urllib.parse
import json
import os
import re

In [10]:
def is_blocked(html: str) -> bool:
    soup = bs4.BeautifulSoup(html, "html.parser")
    text = " ".join(soup.stripped_strings).lower()

    if soup.find(id="captcha-form"):
        return True

    if soup.select_one(
        ".g-recaptcha, iframe[src*='recaptcha'], script[src*='recaptcha']"
    ):
        return True

    if any(
        "/sorry/" in form.get("action", "").lower() for form in soup.find_all("form")
    ):
        return True

    return any(
        marker in text
        for marker in (
            "our systems have detected unusual traffic",
            "computer or network may be sending automated queries",
            "we can't process your request right now",
            "not a robot",
        )
    )


def fetch(url, max_retries=4, browser=False):

    if browser:
        payload = {"url": url, "browserHtml": True}
    else:
        payload = {"url": url, "httpResponseBody": True}

    auth = (os.environ["ZYTE_API_KEY"], "")

    retry_reason = ""

    for attempt in range(max_retries + 1):
        try:
            response = requests.post(
                "https://api.zyte.com/v1/extract", auth=auth, json=payload, timeout=180
            )
            if response.status_code == 200:
                data = response.json()

                if browser:
                    html = data["browserHtml"]
                else:
                    html = base64.b64decode(data["httpResponseBody"]).decode("utf-8")

                if not is_blocked(html):
                    return html
                else:
                    retry_reason = "block or captcha, "
            else:
                retry_reason = f"status = {response.status_code}, type = {response.json()["type"]}, "
                print(response.json())
        except requests.ConnectionError:
            retry_reason = "connection error, "
        except requests.Timeout:
            retry_reason = "timeout, "

        if attempt == max_retries:
            raise RuntimeError("Request failed or CAPTCHA persisted")

        print(f"{retry_reason}retry...")

        time.sleep(random.uniform(2, min(60, 2 ** (attempt + 2))))
    return ""

In [6]:
profile_soup = bs4.BeautifulSoup(
    fetch("https://scholar.google.com/citations?user=qHFA5z4AAAAJ&hl=en&pagesize=100"),
    "html.parser",
)

paper_list = [
    dict(
        {
            "year": row.select_one(".gsc_a_y").get_text(strip=True),
            "citation_url": urllib.parse.urljoin(
                "https://scholar.google.com",
                row.select_one(".gsc_a_at")["href"],
            ),
        }
    )
    for row in profile_soup.select(".gsc_a_tr")
]

print(len(paper_list))
paper_list

57


[{'year': '2023',
  'citation_url': 'https://scholar.google.com/citations?view_op=view_citation&hl=en&user=qHFA5z4AAAAJ&pagesize=100&citation_for_view=qHFA5z4AAAAJ:Y0pCki6q_DkC'},
 {'year': '2020',
  'citation_url': 'https://scholar.google.com/citations?view_op=view_citation&hl=en&user=qHFA5z4AAAAJ&pagesize=100&citation_for_view=qHFA5z4AAAAJ:2osOgNQ5qMEC'},
 {'year': '2020',
  'citation_url': 'https://scholar.google.com/citations?view_op=view_citation&hl=en&user=qHFA5z4AAAAJ&pagesize=100&citation_for_view=qHFA5z4AAAAJ:bnK-pcrLprsC'},
 {'year': '2020',
  'citation_url': 'https://scholar.google.com/citations?view_op=view_citation&hl=en&user=qHFA5z4AAAAJ&pagesize=100&citation_for_view=qHFA5z4AAAAJ:Se3iqnhoufwC'},
 {'year': '2019',
  'citation_url': 'https://scholar.google.com/citations?view_op=view_citation&hl=en&user=qHFA5z4AAAAJ&pagesize=100&citation_for_view=qHFA5z4AAAAJ:IjCSPb-OGe4C'},
 {'year': '2020',
  'citation_url': 'https://scholar.google.com/citations?view_op=view_citation&hl=e

In [ ]:
def parse_citation(html):
    soup = bs4.BeautifulSoup(html, "html.parser")

    title = soup.select_one("#gsc_oci_title")
    fields = {"title": title.get_text(" ", strip=True) if title else None}
    clusters = []

    for row in soup.select("div.gs_scl"):
        key = row.select_one(".gsc_oci_field")
        value = row.select_one(".gsc_oci_value")
        if not key or not value:
            continue

        name = key.get_text(" ", strip=True).lower()

        if name == "scholar articles":
            for snippet in value.select(".gsc_oci_merged_snippet"):
                link = snippet.select_one("a[href*='cluster=']")
                match = (
                    re.search(r"cluster=(\d+)", link.get("href", "")) if link else None
                )
                if match and match.group(1) not in clusters:
                    clusters.append(match.group(1))
        elif name != "total citations" and name != "description":
            fields[name] = value.get_text(" ", strip=True)

    return {**fields, "clusters": clusters}


for paper in paper_list:
    paper |= parse_citation(fetch(paper["citation_url"]))
    print(paper["title"], paper["clusters"])

with open("papers2.json", "w") as file:
    json.dump(paper_list, file, indent=4)

# Clusters

In [11]:
with open("papers2.json", "r") as file:
    paper_list = json.load(file)

In [15]:
def load_clusters(cluster, max_pages=3, num=10):
    for i in range(max_pages):
        soup = bs4.BeautifulSoup(
            fetch(
                f"https://scholar.google.ru/scholar?start={i * num}&num={num}&hl=en&cluster={cluster}"
            )
        )
        print(f"page {i+1}")
        for a in soup.select(".gs_r h3.gs_rt a[href]"):
            print(f"\t{a["href"]}")


load_clusters("10432066948921138844")

retry...


KeyboardInterrupt: 

In [ ]:
for paper in paper_list:
    pass

In [11]:
fetch(
    "https://scholar.google.com/scholar?cluster=10432066948921138844",
    browser=True,
)

{'type': '/download/website-ban', 'title': 'Website Ban', 'status': 520, 'detail': 'Zyte API could not get a ban-free response in a reasonable time. Please, check your URL query parameters. See https://docs.zyte.com/zyte-api/usage/errors.html#zapi-error-url'}
status = 520, type = /download/website-ban, retry...
{'type': '/download/website-ban', 'title': 'Website Ban', 'status': 520, 'detail': 'Zyte API could not get a ban-free response in a reasonable time. Please, check your URL query parameters. See https://docs.zyte.com/zyte-api/usage/errors.html#zapi-error-url'}
status = 520, type = /download/website-ban, retry...


KeyboardInterrupt: 